# Depression dataset, and music

short version of the code

Medical data:

In [ ]:
# https://openneuro.org/datasets/ds003478/versions/1.1.0/download
# James F Cavanagh jcavanagh@unm.edu (2021). EEG: Depression rest. OpenNeuro. [Dataset] doi: 10.18112/openneuro.ds003478.v1.1.0

Musical data:

In [ ]:
# https://www.kunstderfuge.com/

In [ ]:
#pip install mne numpy

# Recurrence 

In [ ]:
# by using the library Pyunicorn, but customising the visualization

import numpy as np
import matplotlib.pyplot as plt
from pyunicorn.timeseries import RecurrencePlot
from tqdm.auto import tqdm
import os
import mne


EPS_std = 0.001
METRIC = "euclidean"


def plot_rp_pyunicorn(series, threshold, time=None, figsize=(6, 6), metric="euclidean"):

    # data preparationPrepare data
    
    ts = np.asarray(series)
    mean = ts.mean()
    std = ts.std()
    
    if std > 0:
        ts = (ts - mean) / std
    else:
        ts = ts - mean  # constant signal safeguard
            
        if std > 0:
            series = (series - mean) / std
        else:
                series = series - mean
    if time is None:
        time = np.arange(len(series))

    # recurrence plot

    rp = RecurrencePlot(
        #series,
        ts,
        metric=metric,
        normalize=True,
        threshold_std=threshold
    )

    R = rp.recurrence_matrix()

    # axes and ticks

    tmin, tmax = time[0], time[-1]
    ticks = np.linspace(tmin, tmax, 6)
    ticklabels = [f"{round(t, 1)}" for t in ticks]

    # plot

    plt.figure(figsize=figsize)

    plt.imshow(
        R,
        cmap="binary",
        origin="lower",
        interpolation="none",
        extent=[tmin, tmax, tmin, tmax],
        vmin=0,
        vmax=1
    )

    plt.xticks(ticks, ticklabels, fontsize=12)
    plt.yticks(ticks, ticklabels, fontsize=12)
    plt.xlabel("Time", fontsize=14)
    plt.ylabel("Time", fontsize=14)

    # Recurrence rate
    rr = R.sum() / R.size
    plt.title(f"Recurrence plot, RR = {rr:.4f}", fontsize=14)

    plt.tight_layout()
    plt.show()
    
    return rp # by adding this, we can still compute laminarity, etc, after showing the plot


# Importing EEG data

In [ ]:
import os
import mne
import numpy as np
from scipy.signal import decimate

# -----------------------
# Paths & subjects
# -----------------------
data_dir = "/Users/mariamannone/Desktop/new_phd_/depression_dataset/dep_data"

subjects = {
    "055": "sub-055_task-Rest_run-01_eeg.set",
    "098": "sub-098_task-Rest_run-01_eeg.set",
}

# -----------------------
# Load data
# -----------------------
raws = {
    sub: mne.io.read_raw_eeglab(f"{data_dir}/{fname}", preload=True)
    for sub, fname in subjects.items()
}

# -----------------------
# Plot (optional)
# -----------------------
for raw in raws.values():
    raw.plot(scalings="auto", n_channels=20, block=True)

# -----------------------
# Save channel-wise time series (FIXED)
# -----------------------
for sub, raw in raws.items():
    data = raw.get_data()          # shape: (n_channels, n_times)
    ch_names = raw.ch_names        # subject-specific

    out_dir = f"timeseries_sub-{sub}"
    os.makedirs(out_dir, exist_ok=True)

    for i, ch in enumerate(ch_names):
        np.save(os.path.join(out_dir, f"sub-{sub}_{ch}.npy"), data[i])

# -----------------------
# Remove first 6 minutes
# -----------------------
def cut_first_minutes(data, sfreq, minutes=6):
    start = int(minutes * 60 * sfreq)
    return data[:, start:]

eeg_cut = {
    sub: cut_first_minutes(
        raw.get_data(),
        raw.info["sfreq"]
    )
    for sub, raw in raws.items()
}


In [ ]:
####

# channel selection

####

idx = [4, 5, 7, 8, 10, 19]
#idx = [3, 4, 5, 7, 9, 19]

for sub, raw in raws.items():
    labels = [raw.ch_names[i] for i in idx if i < len(raw.ch_names)]
    print(f"Subject {sub} channels:", labels)


In [ ]:
def extract_channels_by_index(raw, idx):
    return {
        raw.ch_names[i]: raw.get_data()[i]
        for i in idx
        if i < raw.get_data().shape[0]
    }

ts = {
    sub: extract_channels_by_index(raw, idx)
    for sub, raw in raws.items()
}


ts_55 = ts["055"]
ts_98 = ts["098"]

# channel

In [ ]:
## choice of channel
# channel

ch_idx = 4

# 4, 5, 7, 8, 10, 19

In [ ]:
############
# downsample
############


from scipy.signal import decimate

factor = 10
ch_name = raws["055"].ch_names[ch_idx]

ts_5_55 = ts_55[ch_name]          # 1D array
ts_ds_5_55 = decimate(ts_5_55, factor, ftype="iir")
print(ts_ds_5_55)


In [ ]:
ts_98 = ts["098"]

ch_name = raws["098"].ch_names[ch_idx]

ts_98 = ts_98[ch_name]          # 1D array
ts_ds_5_98 = decimate(ts_98, factor, ftype="iir")
print(ts_ds_5_98)

# Recurrence analysis EEG

In [ ]:
rp_5_55 = plot_rp_pyunicorn(ts_ds_5_55, 0.001, time=None, figsize=(6, 6), metric="euclidean")


laminarity = rp_5_55.laminarity()
determinism = rp_5_55.determinism()
rr = rp_5_55.recurrence_rate()

print("Laminarity (LAM):", laminarity)
print("Determinism (DET):", determinism)
print(f"Recurrence Rate (RR): {rr:.3f}")

In [ ]:
rp_5_98 = plot_rp_pyunicorn(ts_ds_5_98, 0.001, time=None, figsize=(6, 6), metric="euclidean")


laminarity = rp_5_98.laminarity()
determinism = rp_5_98.determinism()
rr = rp_5_98.recurrence_rate()

print("Laminarity (LAM):", laminarity)
print("Determinism (DET):", determinism)
print(f"Recurrence Rate (RR): {rr:.3f}")

# Music analysis

In [ ]:
import pretty_midi
import numpy as np

In [ ]:
def dominant_pitch_per_bin(
    midi_path,
    bin_seconds = 0.01,
    start_time = 0.0,
    end_time = 1 * 60
):
    pm = pretty_midi.PrettyMIDI(midi_path)
    fs = int(1 / bin_seconds)  # frames per second

    # Get piano roll
    piano_roll = pm.get_piano_roll(fs=fs)  # shape (128, T)

    # Convert times (seconds) → frame indices
    start_frame = int(start_time * fs)

    if end_time is not None:
        end_frame = int(end_time * fs)
        piano_roll = piano_roll[:, start_frame:end_frame]
    else:
        piano_roll = piano_roll[:, start_frame:]

    # Dominant pitch per frame
    loudness = piano_roll.max(axis=0)
    pitches = piano_roll.argmax(axis=0).astype(np.int16)

    # Mark silence
    pitches = pitches[pitches >= 0]


    return pitches

Beethoven, 7th Symphony, 4th movement

In [ ]:
pitches = dominant_pitch_per_bin("/Users/mariamannone/Desktop/new_phd_/Belgium_IEEE-conference-template-062824 3/symphony_7_4_(c)cvikl.mid")

Verdi, Ouverture from La Forza del Destino

In [ ]:
#pitches = dominant_pitch_per_bin("/Users/mariamannone/Desktop/Verdi_Wagner/midi_files_Verdi_Wagner/verdi_Part_I.mid")

Tchaikovsky, Swan Lake, 3, Scène

In [ ]:
#pitches = dominant_pitch_per_bin("/Users/mariamannone/Desktop/new_phd_/Belgium_IEEE-conference-template-062824 3/tchaikovsky_swan_lake_03_(c)lucarelli.mid")


In [ ]:
bin_seconds = 0.01
time_1min = np.arange(len(pitches)) * bin_seconds

def remove_repeated_values(x):
    return x[np.insert(x[1:] != x[:-1], 0, True)]

pitches_clean = remove_repeated_values(pitches)
intervals = np.diff(pitches_clean)

#rp_music = plot_rp_pyunicorn(intervals, threshold=0.5)

rp_music = plot_rp_pyunicorn(
    series=intervals,
    threshold=0.1,
    time=time_1min,
    figsize=(6, 6),
    metric="euclidean"
)

laminarity__ =rp_music.laminarity()
determinism__ = rp_music.determinism()
rr__ = rp_music.recurrence_rate()

print(laminarity__)
print(determinism__)
print(rr__)

Uncomment this to store Beethoven's incipt intervals as a time series;
then, comment the first line, go back to the MIDI files above, comment Beethoven
and uncomment Verdi, and so on.

In [ ]:
# change each time
#####
#intervals_Beeth = intervals
#intervals_Verdi = intervals
#intervals_Tchai = intervals

# Introducing a new quantifier


We propose here a density comparison, 
taking into account the difference of sizes of the pairs of recurrence plots (music and EEG).
The new quantifier computes local density maps for the RP of music and the one of EEG.
We resize one of the plots to match the size of the other.
We consider here agglomerate maps rather than one-to-one matrices, for computational reasons.
Thus, this is a coarse-grained approach.
Finally, we obtain a degree of Local Density Similarity (LDS). 

In [ ]:
import numpy as np
from scipy.signal import convolve2d
from skimage.transform import resize
from PIL import Image

In [ ]:
# Coarse-grained downsampling approach

# Computing a density matrix out of a recurrence plot

import numpy as np
from skimage.measure import block_reduce

def coarse_density_map(rp_matrix, grid_size=(1000, 1000)): # 50, 50
    """
    Reduce a recurrence plot to a coarse density map.
    rp_matrix: 2D binary matrix
    grid_size: number of coarse cells (rows, cols)
    """
    n_rows, n_cols = rp_matrix.shape
    # Determine block size
    block_row = max(n_rows // grid_size[0], 1)
    block_col = max(n_cols // grid_size[1], 1)
    # Use block_reduce to compute mean in each block
    density = block_reduce(rp_matrix, block_size=(block_row, block_col), func=np.mean)
    return density

In [ ]:
######
# compute the density similarity,
# taking the max correlation after a choice of different starting points

# 19. Januar

from scipy.signal import correlate2d
# correlate2d computes the cross-correlation between two 2D arrays over all possible shifts
# mode = "full": considers all relative shifts, so D2 can “slide over” D1 in both directions


def density_similarity_shift_invariant(
    rp1,
    rp2,
    grid_size=(100, 100),
    normalize=True
):
    """
    Compare two recurrence plots via coarse density maps,
    allowing for shifts in both directions.

    Returns
    -------
    max_corr : float
        Maximum normalized cross-correlation value
    shift : tuple
        (row_shift, col_shift) that gives max correlation
    """

    D1 = coarse_density_map(rp1, grid_size)
    D2 = coarse_density_map(rp2, grid_size)

    # normalize (important!)
    if normalize:
        D1 = (D1 - D1.mean()) / (D1.std() + 1e-12)
        D2 = (D2 - D2.mean()) / (D2.std() + 1e-12)

    corr = correlate2d(D1, D2, mode="full")

    idx = np.unravel_index(np.argmax(corr), corr.shape)
    max_corr = corr[idx] / D1.size

    # convert index to shift
    shift = (
        idx[0] - (D2.shape[0] - 1),
        idx[1] - (D2.shape[1] - 1)
    )

    return max_corr, shift


#from skimage.metrics import structural_similarity as ssim

#ssim(D1, D2, data_range=D1.max() - D1.min())


# Comparing recurrences of EEGs and music

# With the LDS

In [ ]:
R_music = rp_music.recurrence_matrix()
R_eeg_55   = rp_5_55.recurrence_matrix()
R_eeg_98   = rp_5_98.recurrence_matrix()



dens_inv1 = density_similarity_shift_invariant(R_music, R_eeg_55, grid_size=(50,50), normalize=True)
dens_inv2 = density_similarity_shift_invariant(R_music, R_eeg_98, grid_size=(50,50), normalize=True)

######

# max correlation and shift
print("LDS dens MDD =", dens_inv1)
print("LDS dens normal =", dens_inv2)

In [ ]:
# Beethoven & EEGs

# channel 4
#LDS dens MDD = (0.13500729348708626, (-17, -17))
#LDS dens normal = (0.42892222347107556, (0, 0))

# channel 5
#LDS dens MDD = (0.19369097712269215, (-17, -17))
#LDS dens normal = (0.42892222347107556, (0, 0))

# channel 7
#LDS dens MDD = (0.252754551936789, (-17, -17))
#LDS dens normal = (0.44063364843240976, (5, 5))

# channel 8
#LDS dens MDD = (0.22693436057283703, (-17, -17))
#LDS dens normal = (0.39965599612325453, (5, 5))

# channel 10
#LDS dens MDD = (0.23680210856423034, (-6, -6))
#LDS dens normal = (0.35360362938823464, (-4, -4))

# channel 19
#LDS dens MDD = (0.27141997412058577, (2, 2))
#LDS dens normal = (0.4463294316481113, (1, 1))

############

# Verdi & EEGs

# channel 4
#LDS dens MDD = (0.18047068842796554, (11, 11))
#LDS dens normal = (0.3168121200071857, (3, 3))

# channel 5
#LDS dens MDD = (0.1629258293777443, (16, 16))
#LDS dens normal = (0.3138288601236452, (0, 0))

# channel 7
#LDS dens MDD = (0.1334901287275091, (16, 16))
#LDS dens normal = (0.2625234579272165, (0, 0))

# channel 8
#LDS dens MDD = (0.13162892506674007, (17, 17))
#LDS dens normal = (0.2625234579272165, (0, 0))

# channel 10
#LDS dens MDD = (0.12353272111799188, (-15, -15))
#LDS dens normal = (0.20562782278640704, (-12, -12))

# channel 19
#LDS dens MDD = (0.1645822243433592, (-22, -22))
#LDS dens normal = (0.20562782278640704, (-12, -12))

############

# Tchaik. & EEGs

# channel 4
#LDS dens MDD = (0.14223871571780716, (2, 2))
#LDS dens normal = (0.2923238731402724, (-1, -1))

# channel 5
#LDS dens MDD = (0.14689695648329912, (0, 0))
#LDS dens normal = (0.2503412386943012, (-1, -1))

# channel 7
#LDS dens MDD = (0.13056233005348425, (-23, -23))
#LDS dens normal = (0.2503412386943012, (-1, -1))

# channel 8
#LDS dens MDD = (0.13471884412353805, (2, 2))
#LDS dens normal = (0.2503412386943012, (-1, -1))

# channel 10
#LDS dens MDD = (0.13239092636052213, (-16, -16))
#LDS dens normal = (0.20476374747966314, (-9, -9))

# channel 19
#LDS dens MDD = (0.12827775457439616, (-29, -29))
#LDS dens normal = (0.20476374747966314, (-9, -9))


In [ ]:
# LDS vs MDD
lds_mdd = {
    "Beethoven": {4: 0.13500729348708626, 5: 0.19369097712269215, 7: 0.252754551936789,
                  8: 0.22693436057283703, 10: 0.23680210856423034, 19: 0.27141997412058577},
    "Verdi": {4: 0.18047068842796554, 5: 0.1629258293777443, 7: 0.1334901287275091,
              8: 0.13162892506674007, 10: 0.12353272111799188, 19: 0.1645822243433592},
    "Tchaikovsky": {4: 0.14223871571780716, 5: 0.14689695648329912, 7: 0.13056233005348425,
                    8: 0.13471884412353805, 10: 0.13239092636052213, 19: 0.12827775457439616}
}

# LDS vs Normal
lds_normal = {
    "Beethoven": {4: 0.42892222347107556, 5: 0.42892222347107556, 7: 0.44063364843240976,
                  8: 0.39965599612325453, 10: 0.35360362938823464, 19: 0.4463294316481113},
    "Verdi": {4: 0.3168121200071857, 5: 0.3138288601236452, 7: 0.2625234579272165,
              8: 0.2625234579272165, 10: 0.20562782278640704, 19: 0.20562782278640704},
    "Tchaikovsky": {4: 0.2923238731402724, 5: 0.2503412386943012, 7: 0.2503412386943012,
                    8: 0.2503412386943012, 10: 0.20476374747966314, 19: 0.20476374747966314}
}

channels = [4,5,7,8,10,19]

# Compute Δ LDS per channel and average per piece
for piece in lds_mdd:
    delta_per_channel = [lds_normal[piece][ch] - lds_mdd[piece][ch] for ch in channels]
    delta_mean = sum(delta_per_channel)/len(channels)
    print(f"{piece:12} | ΔLDS per channel: {['%.3f'%d for d in delta_per_channel]} | Average ΔLDS = {delta_mean:.3f}")


# With the array of RQA measurements

In [ ]:
rp = RecurrencePlot(
    #ts_ds_5_55,  # MDD patient
    #ts_ds_5_98,   # normal subject
    intervals,   # music
    metric="euclidean",
    normalize=True,
    threshold_std=EPS_std
)

In [ ]:
RR   = float(rp.recurrence_rate())
DET  = float(rp.determinism())
L    = float(rp.average_diaglength())
ENTR = float(np.mean(rp.diag_entropy()))
LAM  = float(rp.laminarity())
TT   = float(rp.trapping_time())


#ENTR = float(np.mean(ENTR_raw))

rqa_vector = np.array([RR, DET, L, ENTR, LAM, TT])

In [ ]:
print(rqa_vector)

In [ ]:

# channel 4
# MDD
#[0.05266564 0.30553725 2.21339873 0.56408602 0.42042319 2.33979165]
# normal
#[0.00936758 0.24031363 2.15991579 0.465119   0.34368634 2.2493477 ]

# channel 5
# MDD
#[0.05097013 0.29207252 2.19828834 0.53742821 0.40105408 2.30928165]
# normal
#[0.02258871 0.23753978 2.15483435 0.45497264 0.33787906 2.24059244]


# channel 7
# MDD
#[0.07666104 0.32820686 2.23407026 0.59917235 0.4429104  2.37895093]
# normal
#[0.02632825 0.28969756 2.19948567 0.53957144 0.40250697 2.3174287 ]


# channel 8
# MDD
#[0.10132378 0.41104477 2.32793776 0.74157898 0.54539511 2.55487012]
# normal
#[0.03980548 0.33793204 2.24781304 0.62164485 0.46632694 2.41341985]

# channel 10
# MDD
#[0.03961104 0.2755572  2.18361458 0.51040276 0.38341427 2.29772564]
# normal
#[0.01611562 0.28809643 2.19812149 0.5371699  0.3997647  2.31484238]

# channel 19
# MDD
#[0.11183031 0.50906553 2.46632612 0.91474784 0.64972068 2.79496181]
# normal
#[0.05896991 0.41846321 2.3322477  0.74772    0.55308707 2.53301255]




# music:

# Beethoven
#[0.02101806 0.38652243 3.64118896 1.25789171 0.07624633 2.        ]

# Verdi
#[0.02494992 0.39385965 3.03378378 1.30984828 0.04840569 2.        ]

# Tchaikovsky
#[0.02939058 0.25465839 2.7032967  0.82509199 0.17153629 2.        ]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.spatial.distance import euclidean


In [ ]:
import numpy as np

# ---------------------------
# LDS values (LDS vs MDD and Normal)
# ---------------------------
lds_mdd = {
    "Beethoven": [0.1350, 0.1937, 0.2528, 0.2269, 0.2368, 0.2714],
    "Verdi": [0.1805, 0.1629, 0.1335, 0.1316, 0.1235, 0.1646],
    "Tchaikovsky": [0.1422, 0.1469, 0.1306, 0.1347, 0.1324, 0.1283]
}

lds_normal = {
    "Beethoven": [0.4289, 0.4289, 0.4406, 0.3997, 0.3536, 0.4463],
    "Verdi": [0.3168, 0.3138, 0.2625, 0.2625, 0.2056, 0.2056],
    "Tchaikovsky": [0.2923, 0.2503, 0.2503, 0.2503, 0.2048, 0.2048]
}

channels = ["4", "5", "7", "8", "10", "19"]

# ---------------------------
# RQA vectors for EEG (MDD, Normal) and music
# ---------------------------
rqa_eeg_mdd = {
    0: [0.05266564, 0.30553725, 2.21339873, 0.56408602, 0.42042319, 2.33979165],
    1: [0.05097013, 0.29207252, 2.19828834, 0.53742821, 0.40105408, 2.30928165],
    2: [0.07666104, 0.32820686, 2.23407026, 0.59917235, 0.4429104 , 2.37895093],
    3: [0.10132378, 0.41104477, 2.32793776, 0.74157898, 0.54539511, 2.55487012],
    4: [0.03961104, 0.2755572 , 2.18361458, 0.51040276, 0.38341427, 2.29772564],
    5: [0.11183031, 0.50906553, 2.46632612, 0.91474784, 0.64972068, 2.79496181]
}

rqa_eeg_normal = {
    0: [0.00936758, 0.24031363, 2.15991579, 0.465119  , 0.34368634, 2.2493477 ],
    1: [0.02258871, 0.23753978, 2.15483435, 0.45497264, 0.33787906, 2.24059244],
    2: [0.02632825, 0.28969756, 2.19948567, 0.53957144, 0.40250697, 2.3174287 ],
    3: [0.03980548, 0.33793204, 2.24781304, 0.62164485, 0.46632694, 2.41341985],
    4: [0.01611562, 0.28809643, 2.19812149, 0.5371699 , 0.3997647 , 2.31484238],
    5: [0.05896991, 0.41846321, 2.3322477 , 0.74772   , 0.55308707, 2.53301255]
}

rqa_music = {
    "Beethoven": [0.02101806, 0.38652243, 3.64118896, 1.25789171, 0.07624633, 2.0],
    "Verdi": [0.02494992, 0.39385965, 3.03378378, 1.30984828, 0.04840569, 2.0],
    "Tchaikovsky": [0.02939058, 0.25465839, 2.7032967 , 0.82509199, 0.17153629, 2.0]
}

# ---------------------------
# Function to compute Euclidean distance
# ---------------------------
def euclid_dist(a, b):
    return np.linalg.norm(np.array(a) - np.array(b))

# ---------------------------
# Generate table
# ---------------------------
for piece in lds_mdd:
    print(f"--- {piece} ---")
    delta_lds = []
    dist_mdd = []
    dist_normal = []
    for i, ch in enumerate(channels):
        d_lds = lds_normal[piece][i] - lds_mdd[piece][i]
        delta_lds.append(d_lds)
        dm = euclid_dist(rqa_music[piece], rqa_eeg_mdd[i])
        dn = euclid_dist(rqa_music[piece], rqa_eeg_normal[i])
        dist_mdd.append(dm)
        dist_normal.append(dn)
        print(f"{ch:10} | LDS MDD: {lds_mdd[piece][i]:.3f} | LDS Normal: {lds_normal[piece][i]:.3f} | ΔLDS: {d_lds:.3f} | dist_MDD: {dm:.3f} | dist_Normal: {dn:.3f}")
    print(f"Mean ΔLDS: {np.mean(delta_lds):.3f} | Mean dist_MDD: {np.mean(dist_mdd):.3f} | Mean dist_Normal: {np.mean(dist_normal):.3f}\n")


# Lacunarity for EEGs

We use now the definition of lacunarity as provided in:

Braun, Tobias, et al. "Detection of dynamical regime transitions with lacunarity as a multiscale recurrence quantification measure." Nonlinear Dynamics (2021): 1-19.

Original code from https://github.com/ToBraun/RECLAC/blob/main/tutorial.ipynb

In [ ]:
#pip install RECLAC

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
# import both modules from the RECLAC package
import RECLAC.recurrence_plot as rec
import RECLAC.boxcount as bc
fs = 18

In [ ]:
# to be adapted into: on figure for EEGs, one figure for music

In [ ]:
import numpy as np
import RECLAC.boxcount as bc

# recommended...
#dim = 3
#tau = int(raw.info["sfreq"] * 0.02)  # ~20 ms


def compute_lacunarity_curve(
    ts,
    boxes,
    thresh=0.1,
    dim=3,
    tau=1,
    #dim=None,
    #tau=None,
    normalized=True
):
    """
    Compute lacunarity curve Λ(box size) for a 1D EEG time series.
    """

    if dim is None:
        BC = bc.Boxcount(
            ts,
            method='frr',
            thresh=thresh,
            boxes=boxes
        )
    else:
        BC = bc.Boxcount(
            ts,
            method='frr',
            thresh=thresh,
            dim=dim,
            tau=tau,
            boxes=boxes
        )

    _, lac = BC.lacunarity(
        regression=None,
        normalized=normalized
    )

    return lac


In [ ]:
# with the same parameters of the tutorial on lacunarity,
# except the number of boxes, for computational limit

bmin, bmax = 2, 120 # 250
nb = 20

a_boxes = np.logspace(
    np.log10(bmin),
    np.log10(bmax),
    nb,
    dtype=int
)



Channel for lacunarity

In [ ]:
ch_idx = 4  # channel index

factor = 10

ch_name = raws["055"].ch_names[ch_idx]

ts_55 = ts["055"]

ts_5_55 = ts_55[ch_name]          # 1D array
ts_ds_5_55 = decimate(ts_5_55, factor, ftype="iir")
print(ts_ds_5_55)


ch_name = raws["098"].ch_names[ch_idx]


ts_98 = ts["098"]

ch_name = raws["098"].ch_names[ch_idx]
ts_5_98 = ts_98[ch_name]          # 1D array
ts_ds_5_98 = decimate(ts_5_98, factor, ftype="iir")
print(ts_ds_5_98)



In [ ]:
lac_55 = compute_lacunarity_curve(
    ts_ds_5_55,
    boxes=a_boxes,
    thresh=0.001,
    dim=3,
    tau=int(raws["055"].info["sfreq"] * 0.02)
)



lac_98 = compute_lacunarity_curve(
    ts_ds_5_98,
    boxes=a_boxes,
    thresh=0.001,
    dim=3,
    tau=int(raws["055"].info["sfreq"] * 0.02)
)

In [ ]:
# for one single channel

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))

color_dict = {
    "lac_55": "blue",
    "lac_98": "red"
}
    

plt.loglog(
    a_boxes[:len(lac_55)],
    lac_55,
    label="MDD EEG",
    color=color_dict["lac_55"],
    linewidth=2
)

plt.loglog(
    a_boxes[:len(lac_98)],
    lac_98,
    label="Normal EEG",
    color=color_dict["lac_98"],
    linewidth=2
)

plt.xlabel("Box width", fontsize=13)
plt.ylabel(r"$\Lambda$", fontsize=13)
plt.title(f"EEG Lacunarity, channel {ch_idx}", fontsize=14)
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()




# Lacunarity for musical sequences

In [ ]:
# computed separately above

#intervals_Beeth, intervals_Verdi, intervals_Tchai = intervals

In [ ]:
#print(intervals_Beeth)
#print(intervals_Verdi)
#print(intervals_Tchai)

In [ ]:
bmin, bmax = 2, 250
#nb = 100
nb = 100

# changing the number of boxes

a_boxes = np.logspace(
    np.log10(bmin),
    np.log10(bmax),
    nb,
    dtype=int
)


In [ ]:
lac_Beeth = compute_lacunarity_curve(
    intervals_Beeth,
    boxes=a_boxes,
    thresh=0.1
)

lac_Verdi  = compute_lacunarity_curve(
    intervals_Verdi,
    boxes=a_boxes,
    thresh=0.1
)

lac_Tchai = compute_lacunarity_curve(
    intervals_Tchai,
    boxes=a_boxes,
    thresh=0.1
)


In [ ]:
##

# to avoid using empty boxes:

import numpy as np
import matplotlib.pyplot as plt

def plot_lacunarity_curves(a_boxes, lac_dict):
    """
    a_boxes: array of box widths
    lac_dict: dict of name -> lacunarity array (can have different lengths)
    """
    plt.figure(figsize=(8,6))

    for name, lac in lac_dict.items():
        x = a_boxes[:len(lac)]
        y = np.array(lac)
        # Mask any zero or negative values (log scale cannot handle)
        mask = y > 0
        plt.loglog(x[mask], y[mask], label=name, linewidth=2)

    plt.xlabel("Box width", fontsize=13)
    plt.ylabel(r"$\Lambda$", fontsize=13)
    plt.title("Music Lacunarity", fontsize=14)
    plt.grid(True, which="both", alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
lac_dict = {
    "Beethoven": lac_Beeth,
    "Verdi": lac_Verdi,
    "Tchaikovsky": lac_Tchai
}

plot_lacunarity_curves(a_boxes, lac_dict)


In [ ]:
# With the right colors:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_lacunarity_curves(a_boxes, lac_dict, n_box=None):
    """
    Plot multiple lacunarity curves with log-log scale.
    a_boxes: array of box widths
    lac_dict: dict of name -> lacunarity array
    n_box: optional, highlight this box index with a marker
    """
    plt.figure(figsize=(8,6))
    
    # Define specific colors
    color_dict = {
        "Beethoven": "red",
        "Verdi": "green",
        "Tchaikovsky": "blue"
    }
    
    for name, lac in lac_dict.items():
        x = a_boxes[:len(lac)]
        y = np.array(lac)
        mask = y > 0  # mask zeros for log-log
        plt.loglog(x[mask], y[mask], label=name, color=color_dict.get(name, "black"), linewidth=2)
        
        # Optional: mark a specific box
        if n_box is not None and n_box < len(y):
            plt.scatter(x[n_box], y[n_box], color=color_dict.get(name, "black"), s=50)
    
    plt.xlabel("Box width", fontsize=13)
    plt.ylabel(r"$\Lambda$", fontsize=13)
    plt.title("Lacunarity curves of music", fontsize=14)
    plt.grid(True, which="both", alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

# Usage
lac_dict = {
    "Beethoven": lac_Beeth,
    "Verdi": lac_Verdi,
    "Tchaikovsky": lac_Tchai
}

plot_lacunarity_curves(a_boxes, lac_dict)


In [ ]:
# Lacunarity as just one value

In [ ]:
# averaged EEGs and music

In [ ]:
n_box = 30  # the box index we want

# ------------------------------
# Music sequences
# ------------------------------

print("=== Music sequences lacunarity ===")
music_lac = {
    "Beethoven": lac_Beeth,
    "Verdi": lac_Verdi,
    "Tchaikovsky": lac_Tchai
}

for name, lac in music_lac.items():
    if n_box < len(lac):
        print(f"{name}: Λ at box {n_box} = {lac[n_box]:.4f}")
    else:
        print(f"{name}: Λ index {n_box} out of bounds (length = {len(lac)})")


In [ ]:
# ------------------------------
# EEG group curves
# ------------------------------

print("\n=== EEG, one channel ===")


n_box = 19  # box index

# MDD EEG, channel...
if n_box < len(lac_55):
    print(f"MDD EEG, channel {ch_idx}: Λ at box {n_box} = {lac_55[n_box]:.4f}")
else:
    print(f"MDD EEG: Λ index {n_box} out of bounds (length = {len(lac_55)})")

# Normal EEG, channel...
if n_box < len(lac_98):
    print(f"Normal EEG, channel {ch_idx}: Λ at box {n_box} = {lac_98[n_box]:.4f}")
else:
    print(f"Normal EEG: Λ index {n_box} out of bounds (length = {len(lac_98)})")

    

### ALL CHANNELS, averaged

#print("\n=== EEG group lacunarity (averaged channels) ===")
      
           
# Normal EEGs
#if n_box < normal_lac_curves.shape[1]:
#    print(f"Normal EEGs (averaged): Λ at box {n_box} = {normal_lac_curves[:, n_box].mean():.4f}")
#else:
#    print(f"Normal EEGs: Λ index {n_box} out of bounds (length = {normal_lac_curves.shape[1]})")

# MDD EEGs
#if n_box < mdd_lac_curves.shape[1]:
#    print(f"MDD EEGs (averaged): Λ at box {n_box} = {mdd_lac_curves[:, n_box].mean():.4f}")
#else:
#    print(f"MDD EEGs: Λ index {n_box} out of bounds (length = {mdd_lac_curves.shape[1]})")
